In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

!pip install catboost
from catboost import CatBoostClassifier

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.5 MB/s eta 0:00:00


In [ ]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

train = train.dropna(subset=["PitNextLap"])

In [ ]:
print(train.shape)
print(test.shape)

(439140, 16)
(188165, 15)


In [ ]:
target = "PitNextLap"

In [ ]:
def add_features(df):
    df = df.copy()

    df["DegradationRate"] = df["Cumulative_Degradation"] / (df["TyreLife"] + 1)
    df["TyrePressure"] = df["TyreLife"] / (df["LapNumber"] + 1)
    df["IsLateRace"] = (df["RaceProgress"] > 0.7).astype(int)
    df["PositionPressure"] = df["Position"] * df["Position_Change"]
    df["LapEfficiency"] = df["LapTime (s)"] / (df["LapNumber"] + 1)

    return df

train = add_features(train)
test = add_features(test)

In [ ]:
def oof_target_encoding(train, test, col, target, n_splits=5):
    train = train.copy()
    test = test.copy()

    gkf = GroupKFold(n_splits=n_splits)
    groups = train["Race"] + "_" + train["Driver"]

    oof = np.zeros(len(train))
    test_preds = np.zeros(len(test))

    for train_idx, valid_idx in gkf.split(train, train[target], groups):
        tr, val = train.iloc[train_idx], train.iloc[valid_idx]

        means = tr.groupby(col)[target].mean()

        oof[valid_idx] = val[col].map(means)

        test_preds += test[col].map(means).fillna(tr[target].mean()) / n_splits

    # fill unseen categories
    global_mean = train[target].mean()
    oof = np.where(np.isnan(oof), global_mean, oof)
    test_preds = np.where(np.isnan(test_preds), global_mean, test_preds)

    return oof, test_preds

In [ ]:
for col in ["Driver", "Race", "Compound"]:
    print(f"Encoding {col}...")

    train[f"{col}_TE"], test[f"{col}_TE"] = oof_target_encoding(
        train, test, col, target
    )

Encoding Driver...
Encoding Race...
Encoding Compound...


In [ ]:


features = [col for col in train.columns if col not in ["id", target]]

In [ ]:
X = train[features]
y = train[target]
X_test = test[features]

groups = train["Race"] + "_" + train["Driver"]

In [ ]:
gkf = GroupKFold(n_splits=5)

scores = []

for fold, (train_idx, valid_idx) in enumerate(gkf.split(X, y, groups)):
    print(f"\n========== FOLD {fold+1} ==========")

    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    model = CatBoostClassifier(
        iterations=1000,
        learning_rate=0.03,
        depth=8,
        l2_leaf_reg=3,
        eval_metric="AUC",
        verbose=200
    )

    model.fit(
        X_train, y_train,
        eval_set=(X_valid, y_valid),
        cat_features=["Driver", "Compound", "Race"],
        use_best_model=True
    )

    preds = model.predict_proba(X_valid)[:, 1]

    score = roc_auc_score(y_valid, preds)
    print("Fold AUC:", score)

    scores.append(score)


========== FOLD 1 ==========
0:	test: 0.9143761	best: 0.9143761 (0)	total: 1.52s	remaining: 25m 22s
200:	test: 0.9394427	best: 0.9394427 (200)	total: 2m 30s	remaining: 10m
400:	test: 0.9429211	best: 0.9429211 (400)	total: 4m 42s	remaining: 7m 2s
600:	test: 0.9446702	best: 0.9446702 (600)	total: 6m 53s	remaining: 4m 34s
800:	test: 0.9456133	best: 0.9456133 (800)	total: 9m 4s	remaining: 2m 15s
999:	test: 0.9463285	best: 0.9463285 (999)	total: 11m 27s	remaining: 0us

bestTest = 0.9463284544
bestIteration = 999

Fold AUC: 0.946328454422734

========== FOLD 2 ==========
0:	test: 0.9159688	best: 0.9159688 (0)	total: 551ms	remaining: 9m 10s
200:	test: 0.9405190	best: 0.9405190 (200)	total: 2m 17s	remaining: 9m 7s
400:	test: 0.9439423	best: 0.9439423 (400)	total: 4m 32s	remaining: 6m 47s
600:	test: 0.9457608	best: 0.9457608 (600)	total: 6m 46s	remaining: 4m 29s
800:	test: 0.9468442	best: 0.9468442 (800)	total: 9m 9s	remaining: 2m 16s
999:	test: 0.9475135	best: 0.9475135 (999)	total: 11m 22s	r

In [ ]:
print("\n========== RESULTS ==========")
print("Mean AUC:", np.mean(scores))
print("Std AUC:", np.std(scores))


========== RESULTS ==========
Mean AUC: 0.9471448888615154
Std AUC: 0.0007268049995313426


In [ ]:
# remove OOF encoding columns if they exist
for col in ["Driver_TE", "Race_TE", "Compound_TE"]:
    if col in train.columns:
        train.drop(columns=[col], inplace=True)
        test.drop(columns=[col], inplace=True)

print("Cleaned OOF features")

Cleaned OOF features


In [ ]:
def add_interactions(df):
    df = df.copy()

    # Multiplicative interactions
    df["TyreLife_x_Lap"] = df["TyreLife"] * df["LapNumber"]
    df["Degradation_x_Progress"] = df["Cumulative_Degradation"] * df["RaceProgress"]
    df["Position_x_Change"] = df["Position"] * df["Position_Change"]
    df["Stint_x_TyreLife"] = df["Stint"] * df["TyreLife"]
    df["LapTime_x_Degradation"] = df["LapTime (s)"] * df["Cumulative_Degradation"]

    # Ratios
    df["TyreLife_per_Stint"] = df["TyreLife"] / (df["Stint"] + 1)
    df["Lap_per_Progress"] = df["LapNumber"] / (df["RaceProgress"] + 1e-5)
    df["Degradation_per_Lap"] = df["Cumulative_Degradation"] / (df["LapNumber"] + 1)

    return df

In [ ]:
train = add_interactions(train)
test = add_interactions(test)

In [ ]:
train = add_interactions(train)
test = add_interactions(test)

print("Interaction features added")

Interaction features added


In [ ]:
target = "PitNextLap"

In [ ]:
features = [col for col in train.columns if col not in ["id", target]]

print("Total features:", len(features))

Total features: 27


In [ ]:
X = train[features]
y = train[target]
X_test = test[features]

groups = train["Race"] + "_" + train["Driver"]

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

gkf = GroupKFold(n_splits=5)

In [ ]:
from catboost import CatBoostClassifier

scores = []

for fold, (train_idx, valid_idx) in enumerate(gkf.split(X, y, groups)):

    print(f"\n========== FOLD {fold+1} ==========")

    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    model = CatBoostClassifier(
        iterations=1000,
        learning_rate=0.03,
        depth=8,
        l2_leaf_reg=3,
        eval_metric="AUC",
        verbose=200
    )

    model.fit(
        X_train, y_train,
        eval_set=(X_valid, y_valid),
        cat_features=["Driver", "Compound", "Race"],
        use_best_model=True
    )

    preds = model.predict_proba(X_valid)[:, 1]

    score = roc_auc_score(y_valid, preds)
    print("Fold AUC:", score)

    scores.append(score)


========== FOLD 1 ==========
0:	test: 0.9126635	best: 0.9126635 (0)	total: 710ms	remaining: 11m 49s
200:	test: 0.9395055	best: 0.9395055 (200)	total: 2m 17s	remaining: 9m 4s
400:	test: 0.9436453	best: 0.9436453 (400)	total: 4m 20s	remaining: 6m 28s
600:	test: 0.9455481	best: 0.9455481 (600)	total: 6m 36s	remaining: 4m 23s
800:	test: 0.9466734	best: 0.9466734 (800)	total: 8m 43s	remaining: 2m 10s
999:	test: 0.9474456	best: 0.9474456 (999)	total: 10m 53s	remaining: 0us

bestTest = 0.9474456422
bestIteration = 999

Fold AUC: 0.9474456422375115

========== FOLD 2 ==========
0:	test: 0.9129853	best: 0.9129853 (0)	total: 539ms	remaining: 8m 58s
200:	test: 0.9408979	best: 0.9408979 (200)	total: 2m 11s	remaining: 8m 43s
400:	test: 0.9448777	best: 0.9448777 (400)	total: 4m 19s	remaining: 6m 27s
600:	test: 0.9468999	best: 0.9468999 (600)	total: 6m 37s	remaining: 4m 23s
800:	test: 0.9480496	best: 0.9480496 (800)	total: 8m 53s	remaining: 2m 12s
999:	test: 0.9487566	best: 0.9487566 (999)	total: 11

In [ ]:
import numpy as np

print("\n========== RESULTS ==========")
print("Mean AUC:", np.mean(scores))
print("Std AUC:", np.std(scores))


========== RESULTS ==========
Mean AUC: 0.9482270411180206
Std AUC: 0.0008529894267822888


In [ ]:
final_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=3,
    eval_metric="AUC",
    verbose=200
)

final_model.fit(
    X, y,
    cat_features=["Driver", "Compound", "Race"]
)

0:	total: 1.54s	remaining: 25m 42s
200:	total: 3m 13s	remaining: 12m 48s
400:	total: 5m 57s	remaining: 8m 54s
600:	total: 8m 41s	remaining: 5m 46s
800:	total: 11m 27s	remaining: 2m 50s
999:	total: 14m 7s	remaining: 0us


CatBoostClassifier(depth=8, eval_metric='AUC', iterations=1000, l2_leaf_reg=3, learning_rate=0.03, verbose=200)

In [ ]:
test_preds = final_model.predict_proba(X_test)[:, 1]

In [ ]:
submission = pd.DataFrame({
    "id": test["id"],
    "PitNextLap": test_preds
})

In [ ]:
print(submission.shape)
print(submission.head())

(188165, 2)
       id  PitNextLap
0  439140    0.005049
1  439141    0.004192
2  439142    0.005346
3  439143    0.143569
4  439144    0.832494


In [ ]:
submission.to_csv("submission_exp4.csv", index=False)